# Task 1 — Extract: freeCodeCamp

Scrapes AI / Cloud / Data Science articles from freeCodeCamp News using direct search URLs (`?query=ai`, `?query=cloud`, `?query=data`) and saves the result to `data/raw/`.

**Technical note:** Playwright's Sync API cannot run inside Jupyter's own event loop, and on Windows the Async API hits a separate limitation (subprocess creation isn't supported inside the default Jupyter event loop). The fix used here runs the Sync API inside a background `Thread`, which gets its own event loop context — no conflict either way.

## Imports

In [1]:
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
import csv
import time
import re
import datetime
import threading
import asyncio
import sys
import pandas as pd

## Settings

In [2]:
BASE_SEARCH_URL = "https://www.freecodecamp.org/news/search/?query={}"

# One broad search term per topic — trades some recall for speed,
# versus checking all 43 keywords against every article on the full listing.
SEARCH_TERMS = {
    "AI": "ai",
    "Cloud": "cloud",
    "Data Science": "data",
}

CLICKS_PER_TERM = 10
TARGET_ARTICLES_PER_TERM = 150

## Keywords (for enrichment only)

Filtering now happens via the search URL itself. These lists are only used afterwards to fill the `matched_keywords` column for extra detail.

In [3]:
AI_KEYWORDS = [
    "artificial intelligence", "machine learning", "deep learning",
    "generative ai", "generative artificial intelligence",
    "large language model", "large language models", "llm",
    "chatgpt", "openai", "neural network", "neural networks",
    "computer vision", "natural language processing", "nlp"
]

CLOUD_KEYWORDS = [
    "cloud computing", "cloud", "aws", "amazon web services",
    "azure", "microsoft azure", "google cloud",
    "google cloud platform", "gcp", "cloud architecture",
    "cloud services", "cloud storage"
]

DATA_KEYWORDS = [
    "data science", "data analysis", "data analytics",
    "data engineering", "data engineer", "data scientist",
    "data visualization", "big data", "pandas", "numpy", "sql",
    "data pipeline", "data pipelines", "data warehouse",
    "data lake", "machine data"
]

def matches_keyword(text, keyword):
    pattern = r"(?<!\w)" + re.escape(keyword.lower()) + r"(?!\w)"
    return re.search(pattern, text.lower()) is not None

def detect_matched_keywords(title, description, category):
    text = (title + " " + description + " " + category).lower()
    return [kw for kw in AI_KEYWORDS + CLOUD_KEYWORDS + DATA_KEYWORDS
            if matches_keyword(text, kw)]

## Collect article links from a search page

In [4]:
def collect_links(page, search_url, clicks_needed):
    print(f"Opening: {search_url}")
    page.goto(search_url, wait_until="networkidle")

    for i in range(clicks_needed):
        try:
            page.click("#readMoreBtn", timeout=5000)
            print(f"  Clicked 'Load More' ({i + 1}/{clicks_needed})")
            page.wait_for_timeout(2000)
        except Exception:
            print("  No more 'Load More' button — stopping pagination.")
            break

    html = page.content()
    soup = BeautifulSoup(html, "html.parser")
    articles = soup.find_all("article", class_="post-card")
    print(f"  Found {len(articles)} article cards.\n")

    items = []
    for article in articles:
        link_tag = article.find("a", class_="post-card-image-link")
        if link_tag:
            href = link_tag.get("href")
            link = (href if href.startswith("http") else "https://www.freecodecamp.org" + href) if href else None
        else:
            link = None

        # Author name now lives inside a link with class "meta-item"
        # (the old data-test-label="profile-link" moved to the <ul> wrapper, not the link itself)
        author_tag = article.find("a", class_="meta-item")
        author = author_tag.get_text(" ", strip=True) if author_tag else "No author"

        tag_span = article.find("span", class_="post-card-tags")
        category_tag = tag_span.find("a") if tag_span else None
        category = category_tag.get_text(" ", strip=True) if category_tag else "No category"

        title_tag = article.find("h2") or article.find("h3")
        listing_title = title_tag.get_text(" ", strip=True) if title_tag else ""

        description_tag = article.find("p")
        listing_description = description_tag.get_text(" ", strip=True) if description_tag else ""

        if link:
            items.append({
                "url": link, "author": author, "category": category,
                "listing_title": listing_title, "listing_description": listing_description
            })

    return items

## Scrape a single article page for full details

In [5]:
def scrape_article(page, url, listing_author, listing_category, primary_topic):
    page.goto(url, wait_until="networkidle")
    html = page.content()
    soup = BeautifulSoup(html, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(" ", strip=True) if title_tag else "No title"

    pub_date_tag = soup.find("meta", attrs={"property": "article:published_time"})
    pub_date = pub_date_tag.get("content") if pub_date_tag else "No date"

    desc_tag = soup.find("meta", attrs={"name": "description"})
    description = desc_tag.get("content") if desc_tag else "No description"

    matched_keywords = detect_matched_keywords(title, description, listing_category)

    return {
        "source": "freeCodeCamp", "category": listing_category, "title": title,
        "author": listing_author, "publication_date": pub_date,
        "description": description, "url": url, "topic": primary_topic,
        "matched_keywords": ", ".join(matched_keywords)
    }

## Run the scraper (inside a background thread)

This is the workaround that already proved to work on this machine: running the Sync API inside a `Thread` gives it a fresh context that doesn't collide with Jupyter's own event loop, on Windows or otherwise.

In [6]:
results = []
scraper_error = None

def run_scraper():
    global results, scraper_error
    # Playwright launches the browser as a subprocess. On Windows only the
    # Proactor event loop supports subprocesses (Jupyter defaults to Selector).
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        asyncio.set_event_loop(asyncio.new_event_loop())

    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()

            seen_urls = set()

            for topic, term in SEARCH_TERMS.items():
                search_url = BASE_SEARCH_URL.format(term)
                items = collect_links(page, search_url, CLICKS_PER_TERM)

                collected_for_term = 0
                for item in items:
                    if collected_for_term >= TARGET_ARTICLES_PER_TERM:
                        break
                    if item["url"] in seen_urls:
                        continue
                    seen_urls.add(item["url"])

                    try:
                        data = scrape_article(page, item["url"], item["author"], item["category"], topic)
                        results.append(data)
                        collected_for_term += 1
                        print(f"  [{topic}] [{collected_for_term}/{TARGET_ARTICLES_PER_TERM}] {data['title']}")
                    except Exception as e:
                        print(f"  [{topic}] Failed: {item['url']} - {e}")

                    time.sleep(1)

            browser.close()
    except Exception as e:
        # Exceptions inside a thread are otherwise lost - keep it to re-raise below
        scraper_error = e


scraper_thread = threading.Thread(target=run_scraper)
scraper_thread.start()
scraper_thread.join()

if scraper_error:
    raise scraper_error

print(f"\nDone scraping. {len(results)} total articles collected across {len(SEARCH_TERMS)} search terms.")

Opening: https://www.freecodecamp.org/news/search/?query=ai
  Clicked 'Load More' (1/10)
  Clicked 'Load More' (2/10)
  Clicked 'Load More' (3/10)
  Clicked 'Load More' (4/10)
  Clicked 'Load More' (5/10)
  Clicked 'Load More' (6/10)
  Clicked 'Load More' (7/10)
  Clicked 'Load More' (8/10)
  Clicked 'Load More' (9/10)
  Clicked 'Load More' (10/10)
  Found 275 article cards.

  [AI] [1/150] How to Build an AI Chat App Interface With the Vercel AI SDK and Shadcn/ui
  [AI] [2/150] How to Build a Self-Evaluating AI System: Automated Testing and Evaluation Pipelines for LLM Applications
  [AI] [3/150] How AI Is Changing Malware Detection: From Traditional Antivirus to Next-Gen Protection
  [AI] [4/150] How to Build an AI Chatbot with Gemini and Vercel Serverless Functions 🚀
  [AI] [5/150] How AI Receptionists Work: The Architecture Behind AI Phone Agents
  [AI] [6/150] How AI Is Changing Patching and What Devs Need to Know About Exposure Management
  [AI] [7/150] Agentic AI Engineering in 

## Save to `data/raw/`

In [7]:
fieldnames = ["source", "category", "title", "author", "publication_date",
              "description", "url", "topic", "matched_keywords"]

today = datetime.date.today().isoformat()
output_file = f"../data/raw/freecodecamp_{today}.csv"

with open(output_file, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} articles")
print(f"File: {output_file}")

Saved 450 articles
File: ../data/raw/freecodecamp_2026-09-14.csv


## Preview the result

In [8]:
df = pd.read_csv(output_file)
print(f"Rows: {len(df)}")
print(f"Articles per topic:\n{df['topic'].value_counts()}")
df.head()

Rows: 450
Articles per topic:
topic
AI              150
Cloud           150
Data Science    150
Name: count, dtype: int64


,source,category,title,author,publication_date,description,url,topic,matched_keywords
0,freeCodeCamp,#shadcn ui,How to Build an AI Chat App Interface With the...,Vaibhav Gupta,2026-09-11T16:21:29.864Z,Every other AI product you open today has the ...,https://www.freecodecamp.org/news/how-to-build...,AI,NaN
1,freeCodeCamp,#Artificial Intelligence,How to Build a Self-Evaluating AI System: Auto...,Jude Otine,2026-09-11T15:24:04.941Z,So you shipped your AI feature and it works in...,https://www.freecodecamp.org/news/build-a-self...,AI,"artificial intelligence, llm"
2,freeCodeCamp,#Security,How AI Is Changing Malware Detection: From Tra...,Manish Shivanandhan,2026-09-11T15:22:46.931Z,Malware used to be simple to describe. A virus...,https://www.freecodecamp.org/news/how-ai-is-ch...,AI,NaN
3,freeCodeCamp,#AI,How to Build an AI Chatbot with Gemini and Ver...,Johnson Samuel,2026-09-07T22:35:39.060Z,"A couple of months back, I built a chatbot app...",https://www.freecodecamp.org/news/how-to-build...,AI,NaN
4,freeCodeCamp,#AI,How AI Receptionists Work: The Architecture Be...,Manish Shivanandhan,2026-09-04T20:07:46.216Z,An AI receptionist may sound simple from the o...,https://www.freecodecamp.org/news/how-ai-recep...,AI,NaN
